# Zero-Star Exploit Detection Via Memory Forensics

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import MinMaxScaler
import struct
import re
from collections import defaultdict

class ZeroDayExploitDetector:
    def __init__(self):
        self.svm_model = OneClassSVM(nu=0.05, kernel='rbf', gamma='scale')
        self.scaler = MinMaxScaler()
        self.shellcode_patterns = [rb'\x90{4,}', rb'\xeb\x0c', rb'\x31\xc0', rb'\x50\x68']

    def extract_memory_features(self, memory_dump):
        executable_regions = self.find_executable_regions(memory_dump)
        features = []
        for region in executable_regions:
            entropy = self.calculate_entropy(region)
            shellcode_score = self.detect_shellcode_patterns(region)
            rop_score = self.detect_rop_chains(region)
            control_flow_anomalies = self.analyze_control_flow(region)
            features.append([entropy, shellcode_score, rop_score, control_flow_anomalies])
        return np.array(features) if features else np.array([[0, 0, 0, 0]])

    def find_executable_regions(self, memory_dump):
        regions = []
        for i in range(0, len(memory_dump) - 4096, 4096):
            chunk = memory_dump[i:i+4096]
            opcodes = [b'\x48\x89', b'\x48\x8b', b'\xe8', b'\xc3', b'\x90']
            if sum(chunk.count(opcode) for opcode in opcodes) > 5:
                regions.append(chunk)
        return regions

    def detect_shellcode_patterns(self, data):
        score = sum(len(re.findall(pattern, data)) for pattern in self.shellcode_patterns)
        return min(score, 10) / 10.0

    def detect_rop_chains(self, data):
        gadget_pattern = rb'\xc3|\xc2.{2}|\xff\xe0|\xff\xe4|\xff\xe6'
        gadgets = re.findall(gadget_pattern, data)
        return min(len(gadgets), 20) / 20.0

    def analyze_control_flow(self, data):
        jumps = len(re.findall(rb'\xe9.{4}|\xeb.|\x0f\x8.', data))
        calls = len(re.findall(rb'\xe8.{4}|\xff\x15', data))
        returns = data.count(b'\xc3')
        return min((jumps + calls) / max(returns, 1), 5.0) / 5.0

    def calculate_entropy(self, data):
        if not data: return 0
        byte_counts = np.bincount(np.frombuffer(data, dtype=np.uint8))
        probs = byte_counts / len(data)
        return -np.sum(probs * np.log2(probs + 1e-10)) / 8.0

    def train_baseline(self, benign_memory_dumps):
        features = [f for dump in benign_memory_dumps for f in self.extract_memory_features(dump)]
        features_scaled = self.scaler.fit_transform(features)
        self.svm_model.fit(features_scaled)

    def detect_zero_day(self, memory_dump):
        features = self.extract_memory_features(memory_dump)
        features_scaled = self.scaler.transform(features)
        predictions = self.svm_model.predict(features_scaled)
        return np.mean(predictions == -1) > 0.3